
# Roxy notebook example: k-mers over full amino acid alphabet

This notebook is a **reference implementation example** for the **k-mer descriptor family (full alphabet)** in Roxy.

k-mer descriptors generalize AAC (k=1) and DPC (k=2) to arbitrary k, capturing **local sequence patterns of increasing complexity**.

## Covered outputs

This notebook implements:

- sequence cleaning
- generation of all possible k-mers (k=1,2,3)
- absolute k-mer counts
- normalized k-mer frequencies
- configurable k values
- handling of short sequences
- sparse inspection (non-zero k-mers)
- dataset-level summaries
- class-style implementation for integration into Roxy

⚠️ Note:
- k=3 produces **8000 features**, so use carefully.
- This notebook demonstrates k=1,2,3 but emphasizes k=1 and k=2 for practical use.


In [1]:

from collections import Counter
from itertools import product

import numpy as np
import pandas as pd


## Demo dataset

In [2]:

df_demo = pd.DataFrame(
    {
        "sequence_id": [
            "kmer_1", "kmer_2", "kmer_3", "kmer_4"
        ],
        "sequence": [
            "MKWVTFISLLFLFSSAYSRGVFRR",
            "GGGGGGGGGGGGGGG",
            "KRRKRRKRRKRRDDDDEE",
            "ACDEFGHIKLMNPQRSTVWY",
        ],
    }
)

df_demo


,sequence_id,sequence
0,kmer_1,MKWVTFISLLFLFSSAYSRGVFRR
1,kmer_2,GGGGGGGGGGGGGGG
2,kmer_3,KRRKRRKRRKRRDDDDEE
3,kmer_4,ACDEFGHIKLMNPQRSTVWY


## Constants

In [3]:

STANDARD_AA = list("ACDEFGHIKLMNPQRSTVWY")
STANDARD_AA_SET = set(STANDARD_AA)


## Helper functions

In [4]:

def clean_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper().replace("*", "")
    return "".join([aa for aa in seq if aa in STANDARD_AA_SET])


def generate_all_kmers(k: int):
    return ["".join(p) for p in product(STANDARD_AA, repeat=k)]


def extract_kmers(seq: str, k: int):
    seq = clean_sequence(seq)
    if len(seq) < k:
        return []
    return [seq[i:i+k] for i in range(len(seq) - k + 1)]


def kmer_count_dict(seq: str, k: int, all_kmers):
    kmers = extract_kmers(seq, k)
    counts = Counter(kmers)
    return {f"k{k}_count_{km}": counts.get(km, 0) for km in all_kmers}


def kmer_frequency_dict(seq: str, k: int, all_kmers):
    kmers = extract_kmers(seq, k)
    total = len(kmers)
    if total == 0:
        return {f"k{k}_freq_{km}": np.nan for km in all_kmers}
    counts = Counter(kmers)
    return {f"k{k}_freq_{km}": counts.get(km, 0) / total for km in all_kmers}


## Core k-mer descriptor function

In [5]:

def kmer_descriptors(seq: str, k: int, include_counts=True, include_freq=True):
    seq = clean_sequence(seq)
    all_kmers = generate_all_kmers(k)
    observed = extract_kmers(seq, k)

    out = {
        f"k{k}_length": len(seq),
        f"k{k}_total_kmers": len(observed),
        f"k{k}_unique_kmers": len(set(observed)),
    }

    if include_counts:
        out.update(kmer_count_dict(seq, k, all_kmers))
    if include_freq:
        out.update(kmer_frequency_dict(seq, k, all_kmers))

    if include_freq and len(observed) > 0:
        out[f"k{k}_freq_sum"] = sum(out[f"k{k}_freq_{km}"] for km in all_kmers)
    else:
        out[f"k{k}_freq_sum"] = np.nan

    return out


## Example: k=2 (DPC equivalent)

In [6]:

example = kmer_descriptors(df_demo.loc[0, "sequence"], k=2)
list(example.items())[:10]


[('k2_length', 24),
 ('k2_total_kmers', 23),
 ('k2_unique_kmers', 22),
 ('k2_count_AA', 0),
 ('k2_count_AC', 0),
 ('k2_count_AD', 0),
 ('k2_count_AE', 0),
 ('k2_count_AF', 0),
 ('k2_count_AG', 0),
 ('k2_count_AH', 0)]

## Apply k=1, k=2, k=3

In [7]:

df_k1 = df_demo["sequence"].apply(lambda x: kmer_descriptors(x, k=1)).apply(pd.Series)
df_k2 = df_demo["sequence"].apply(lambda x: kmer_descriptors(x, k=2)).apply(pd.Series)
df_k3 = df_demo["sequence"].apply(lambda x: kmer_descriptors(x, k=3)).apply(pd.Series)

df_k1.shape, df_k2.shape, df_k3.shape


((4, 44), (4, 804), (4, 16004))

## Merge k=1 and k=2 (recommended)

In [8]:

df_kmer = pd.concat([df_demo, df_k1, df_k2], axis=1)
df_kmer.head()


,sequence_id,sequence,k1_length,k1_total_kmers,k1_unique_kmers,k1_count_A,k1_count_C,k1_count_D,k1_count_E,k1_count_F,...,k2_freq_YN,k2_freq_YP,k2_freq_YQ,k2_freq_YR,k2_freq_YS,k2_freq_YT,k2_freq_YV,k2_freq_YW,k2_freq_YY,k2_freq_sum
0,kmer_1,MKWVTFISLLFLFSSAYSRGVFRR,24.0,24.0,13.0,1.0,0.0,0.0,0.0,4.0,...,0.0,0.0,0.0,0.0,0.043478,0.0,0.0,0.0,0.0,1.0
1,kmer_2,GGGGGGGGGGGGGGG,15.0,15.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1.0
2,kmer_3,KRRKRRKRRKRRDDDDEE,18.0,18.0,4.0,0.0,0.0,4.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1.0
3,kmer_4,ACDEFGHIKLMNPQRSTVWY,20.0,20.0,20.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1.0


## Sparse inspection (non-zero k-mers)

In [9]:

k2_freq_cols = [c for c in df_k2.columns if c.startswith("k2_freq_")]

row = df_k2.loc[0, k2_freq_cols]
non_zero = row[row > 0].sort_values(ascending=False)

non_zero.head(15)


k2_freq_sum    1.000000
k2_freq_LF     0.086957
k2_freq_AY     0.043478
k2_freq_FR     0.043478
k2_freq_FS     0.043478
k2_freq_FI     0.043478
k2_freq_FL     0.043478
k2_freq_IS     0.043478
k2_freq_GV     0.043478
k2_freq_LL     0.043478
k2_freq_KW     0.043478
k2_freq_RG     0.043478
k2_freq_RR     0.043478
k2_freq_SA     0.043478
k2_freq_MK     0.043478
Name: 0, dtype: float64

## Dataset-level summary

In [10]:

summary_k2 = (
    df_k2[k2_freq_cols]
    .mean(axis=0)
    .sort_values(ascending=False)
    .head(15)
)

summary_k2


k2_freq_sum    1.000000
k2_freq_GG     0.250000
k2_freq_RR     0.069693
k2_freq_KR     0.058824
k2_freq_RK     0.044118
k2_freq_DD     0.044118
k2_freq_DE     0.027864
k2_freq_LF     0.021739
k2_freq_RD     0.014706
k2_freq_EE     0.014706
k2_freq_PQ     0.013158
k2_freq_CD     0.013158
k2_freq_RS     0.013158
k2_freq_ST     0.013158
k2_freq_VW     0.013158
dtype: float64

## Sanity checks

In [11]:

assert "k2_total_kmers" in df_k2.columns
valid_rows = df_k2["k2_total_kmers"] > 0
assert np.allclose(df_k2.loc[valid_rows, "k2_freq_sum"], 1.0)

print("k-mer checks passed")


k-mer checks passed


## Class-style implementation

In [12]:

class KmerDescriptors:

    def __init__(self, k=2, include_counts=True, include_freq=True):
        self.k = k
        self.include_counts = include_counts
        self.include_freq = include_freq
        self.all_kmers = generate_all_kmers(k)

    def transform_sequence(self, seq):
        return kmer_descriptors(
            seq,
            k=self.k,
            include_counts=self.include_counts,
            include_freq=self.include_freq
        )

    def transform(self, sequences):
        return pd.DataFrame([self.transform_sequence(s) for s in sequences])


kmer_transformer = KmerDescriptors(k=2)
kmer_matrix = kmer_transformer.transform(df_demo["sequence"])
kmer_matrix.head()


,k2_length,k2_total_kmers,k2_unique_kmers,k2_count_AA,k2_count_AC,k2_count_AD,k2_count_AE,k2_count_AF,k2_count_AG,k2_count_AH,...,k2_freq_YN,k2_freq_YP,k2_freq_YQ,k2_freq_YR,k2_freq_YS,k2_freq_YT,k2_freq_YV,k2_freq_YW,k2_freq_YY,k2_freq_sum
0,24,23,22,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.043478,0.0,0.0,0.0,0.0,1.0
1,15,14,1,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1.0
2,18,17,7,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1.0
3,20,19,19,0,1,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1.0


## Optional export

In [ ]:
# df_kmer.to_csv("demo_kmer_descriptors.csv", index=False)
